# 🚀 LangChain - Gestion de la Mémoire : Patterns **Modernes** (LangChain 1.x)

> **Stack 100% Locale** | Ollama `gemma3:4b` · HuggingFace Embeddings · ChromaDB  
> **Zéro classe dépréciée** — Ce notebook utilise exclusivement les API stables de LangChain 1.x.

---

## 📋 Sommaire des 9 Concepts + 5 Avancés

| # | Concept | API clé |
|---|---------|---------|
| 1 | **Pattern fondamental LCEL** | `InMemoryChatMessageHistory` + `RunnableWithMessageHistory` |
| 2 | **Fenêtre glissante** | `trim_messages(strategy="last")` |
| 3 | **Budget de tokens** | `trim_messages(max_tokens=N)` |
| 4 | **Résumé automatique** | LCEL custom summarization |
| 5 | **Mémoire sémantique** | ChromaDB retriever dans LCEL |
| 6 | **État structuré (Entités)** | `LangGraph StateGraph` + `TypedDict` |
| 7 | **Graphe de connaissances** | `LangGraph` avec facts graph |
| 8 | **Multi-sessions** | Store dict + `session_id` routing |
| 9 | **Persistance cross-session** | `LangGraph InMemorySaver` + `thread_id` |
| 10 | **Prompt-as-Memory** (avancé) | Mécanique du contexte |
| 11 | **Threads & Checkpointers** (avancé) | `InMemorySaver`, `SqliteSaver` |
| 12 | **Streamlit + session_state** (avancé) | Interface persistante |
| 13 | **Limites pratiques** (avancé) | Context window & hallucinations |
| 14 | **Stratégies de production** (avancé) | Sliding window + summarization |


## ⚙️ Installation

Toutes les dépendances sont dans votre `requirements.txt`. Rien de nouveau à installer.


In [1]:
# Vérification que les packages clés sont bien installés
import importlib, sys

packages = [
    "langchain_core",
    "langchain_ollama",
    "langchain_chroma",
    "langchain_huggingface",
    "langgraph",
]

all_ok = True
for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg} — pip install {pkg.replace('_', '-')}")
        all_ok = False

print()
print("✅ Stack 100% moderne prête !" if all_ok else "❌ Installer les packages manquants.")


  ✅ langchain_core
  ✅ langchain_ollama
  ✅ langchain_chroma
  ✅ langchain_huggingface
  ✅ langgraph

✅ Stack 100% moderne prête !


## 🔧 Setup Commun

Ce bloc est exécuté **une seule fois** en début de session. Il initialise :
- Le **LLM local** via Ollama (`ChatOllama` — interface chat)
- Les **embeddings** HuggingFace (pour la mémoire sémantique, concept 5)


In [2]:
# ─── Imports communs à tous les concepts ─────────────────────────────
from langchain_ollama import ChatOllama                        # LLM chat local
from langchain_huggingface import HuggingFaceEmbeddings        # Embeddings locaux
from langchain_core.output_parsers import StrOutputParser      # Parser texte simple
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# LLM local — gemma3:4b via Ollama
# Assurez-vous qu'Ollama est démarré : `ollama serve`
llm = ChatOllama(model="gemma3:4b")

# Embeddings locaux — modèle léger, ~22MB
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("✅ LLM et embeddings initialisés.")
print(f"   Modèle LLM   : {llm.model}")
print(f"   Modèle embeds: all-MiniLM-L6-v2")


✅ LLM et embeddings initialisés.
   Modèle LLM   : gemma3:4b
   Modèle embeds: all-MiniLM-L6-v2


---
## 🧩 Concept 1 — Pattern Fondamental LCEL

### 📖 Théorie

Le pattern de base de la mémoire moderne en LangChain 1.x repose sur **deux briques** :

1. **`InMemoryChatMessageHistory`** — un simple tableau de messages (`HumanMessage`, `AIMessage`) stocké en RAM.  
2. **`RunnableWithMessageHistory`** — un décorateur LCEL qui **lit** l'historique avant chaque invocation et **écrit** la nouvelle paire question/réponse après.

```
User Input ──► [Historique] ──► Prompt ──► LLM ──► Réponse
                  ▲                                    │
                  └────────────────────────────────────┘
                              (auto-save)
```

> **💡 Trainer's Note** : `RunnableWithMessageHistory` remplace **à la fois** `ConversationChain` ET `ConversationBufferMemory`. Plus besoin d'instancier deux objets séparés.

### 🔑 Paramètres clés
- `input_messages_key` : clé du dict d'entrée contenant le message humain
- `history_messages_key` : clé du `MessagesPlaceholder` dans le prompt
- `configurable` → `session_id` : identifiant de la session (un store = une session)


In [4]:
# ─── Concept 1 : Pattern fondamental LCEL ────────────────────────────

# 1. Store en mémoire (dict) : session_id → historique
#    Chaque clé = un utilisateur ou une conversation distincte
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Retourne (ou crée) l'historique pour un session_id donné."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 2. Prompt avec placeholder pour l'historique
prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant support technique. Réponds de façon concise en français."),
    MessagesPlaceholder(variable_name="history"),  # ← injecte l'historique ici
    ("human", "{input}"),                          # ← message courant de l'utilisateur
])

# 3. Chaîne LCEL pure (aucune classe dépréciée)
chain = prompt | llm | StrOutputParser()

# 4. Décorateur d'historique : lit + écrit l'historique automatiquement
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,          # fonction qui retourne l'historique
    input_messages_key="input",   # clé du message humain dans l'input
    history_messages_key="history", # clé du MessagesPlaceholder
)

# ─── Test : simulation d'une conversation de support ─────────────────
config = {"configurable": {"session_id": "support_001"}}

print("=== Conversation Support (session: support_001) ===\n")

r1 = chain_with_history.invoke({"input": "Mon imprimante ne répond plus depuis ce matin."}, config=config) # type: ignore
print(" Utilisateur : Mon imprimante ne répond plus depuis ce matin.")
print(f" Assistant   : {r1}\n")

r2 = chain_with_history.invoke({"input": "Est-ce que redémarrer le spouleur d'impression pourrait aider ?"}, config=config) # type: ignore
print(" Utilisateur : Est-ce que redémarrer le spouleur d'impression pourrait aider ?")
print(f" Assistant   : {r2}\n")

r3 = chain_with_history.invoke({"input": "Et si ça ne marche pas, que dois-je faire ?"}, config=config) # type: ignore
print(" Utilisateur : Et si ça ne marche pas, que dois-je faire ?")
print(f" Assistant   : {r3}\n")

# ─── Inspection de l'historique stocké ───────────────────────────────
print("=== Historique stocké (store['support_001']) ===")
for msg in store["support_001"].messages:
    role = "[H]" if isinstance(msg, HumanMessage) else "[S]]"
    print(f"{role} {msg.content[:80]}...")


/usr/local/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


=== Conversation Support (session: support_001) ===

 Utilisateur : Mon imprimante ne répond plus depuis ce matin.
 Assistant   : Pouvez-vous me donner le modèle de votre imprimante et le système d'exploitation de votre ordinateur ?


 Utilisateur : Est-ce que redémarrer le spouleur d'impression pourrait aider ?
 Assistant   : Oui, cela vaut la peine d'essayer. Essayez de redémarrer le service "Publication réseau" (Windows) ou "Services d'impression" (macOS).


 Utilisateur : Et si ça ne marche pas, que dois-je faire ?
 Assistant   : Vérifiez les câbles, essayez une autre prise, puis réinstallez les pilotes de l'imprimante.


=== Historique stocké (store['support_001']) ===
[H] Mon imprimante ne répond plus depuis ce matin....
[S]] Pouvez-vous me donner le modèle de votre imprimante et le système d'exploitation...
[H] Est-ce que redémarrer le spouleur d'impression pourrait aider ?...
[S]] Oui, cela vaut la peine d'essayer. Essayez de redémarrer le service "Publication...
[H] Et si ça n

---
## 🧩 Concept 2 — Fenêtre Glissante (`trim_messages`)

### 📖 Théorie

La fenêtre glissante résout le problème du **contexte qui grandit à l'infini**. On ne garde que les **K derniers messages**, en rejetant les plus anciens.

```
Avant trim : [M1, M2, M3, M4, M5, M6] ← 6 messages
             ─────────────── ↑ max_messages=4
Après trim : [        M3, M4, M5, M6]  ← 4 messages conservés
```

**`trim_messages`** est la fonction moderne (dans `langchain_core.messages`). Elle remplace `ConversationBufferWindowMemory`.

> **💡 Trainer's Note** : La subtilité est d'appliquer le trim **avant** que les messages soient injectés dans le prompt, mais **après** que l'historique complet a été chargé du store. On le fait avec `RunnablePassthrough.assign()`.


In [7]:
# ─── Concept 2 : Fenêtre Glissante ──────────────────────────────────

from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough

# Store dédié à ce concept
store_window = {}

def get_history_window(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store_window:
        store_window[session_id] = InMemoryChatMessageHistory()
    return store_window[session_id]

# Trimmer : garde les 4 derniers messages (= 2 échanges humain/IA)
# strategy="last" = on garde les plus récents
# token_counter=len = compte par NOMBRE de messages (pas de tokens)
# start_on="human" = la fenêtre commence toujours sur un message humain
trimmer = trim_messages(
    max_tokens=4,
    strategy="last",
    token_counter=len,    # len() → compte le nombre de messages
    include_system=True,  # garde le message système s'il y en a un
    allow_partial=False,
    start_on="human",     # assure qu'on commence par un message humain
    
)

prompt_window = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant support. Tu gardes uniquement les 2 derniers échanges."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# Clé : on applique le trimmer sur l'historique AVANT injection dans le prompt
chain_window = (
    RunnablePassthrough.assign(
        history=lambda x: trimmer.invoke(x["history"])  # trim avant prompt
    )
    | prompt_window
    | llm
    | StrOutputParser()
)

chain_with_window = RunnableWithMessageHistory(
    chain_window,
    get_history_window,
    input_messages_key="input",
    history_messages_key="history",
)

# ─── Test : 5 échanges, mais le LLM ne "voit" que les 2 derniers ────
config_w = {"configurable": {"session_id": "window_001"}}

questions = [
    "Mon PC bug depuis hier soir.",
    "Il s'éteint aléatoirement toutes les 30 minutes.",
    "La température CPU est à 95°C.",
    "J'ai nettoyé les ventilateurs, ça n'a pas changé.",
    "Quelle est la prochaine étape selon toi ?",
]

print("=== Fenêtre glissante (max 2 échanges en contexte) ===\n")
for q in questions:
    r = chain_with_window.invoke({"input": q}, config=config_w) # type: ignore
    print(f"[H] {q}")
    print(f"[S] {r[:150]}...\n")

print(f"Historique COMPLET stocké : {len(store_window['window_001'].messages)} messages")
print("(Le LLM n'en voit que 4 = 2 derniers échanges, mais tout est conservé en store)")


/usr/local/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


=== Fenêtre glissante (max 2 échanges en contexte) ===

[H] Mon PC bug depuis hier soir.
[S] Je suis désolé d'apprendre que ton PC bug ! Pourrais-tu me donner plus de détails sur ce qui se passe ? Par exemple :

*   Quels sont les symptômes qu...

[H] Il s'éteint aléatoirement toutes les 30 minutes.
[S] Je suis désolé d'apprendre que ton PC s'éteint aléatoirement toutes les 30 minutes. C'est un problème sérieux. 

Pourrais-tu me donner quelques inform...

[H] La température CPU est à 95°C.
[S] Je comprends. Une température de CPU à 95°C est effectivement un problème majeur et peut expliquer les extinctions aléatoires.

Pourrais-tu me dire si...

[H] J'ai nettoyé les ventilateurs, ça n'a pas changé.
[S] D'accord. Le nettoyage des ventilateurs n'ayant pas résolu le problème, il est fort probable que le problème soit lié à un manque de refroidissement o...

[H] Quelle est la prochaine étape selon toi ?
[S] Selon moi, la prochaine étape serait de vérifier et, si nécessaire, de remplacer la

---
## 🧩 Concept 3 — Budget de Tokens (`trim_messages` avec `max_tokens`)

### 📖 Théorie

Plutôt que de compter les **messages**, on peut limiter par **nombre de tokens**. C'est plus précis car un message peut être très court ou très long.

**Problème avec les LLMs locaux** : `ChatOllama` n'implémente pas `get_num_tokens()`. On utilise une **fonction d'approximation** : ~4 caractères = 1 token (règle empirique).

```python
def approx_tokens(messages) -> int:
    return sum(len(str(m.content)) // 4 for m in messages)
```

> **💡 Trainer's Note** : En production avec un LLM cloud, on passerait `token_counter=llm` directement. En local avec Ollama, l'approximation `//4` est suffisante pour une formation. L'enjeu pédagogique est le concept, pas la précision au token près.


In [ ]:
# ─── Concept 3 : Budget de Tokens ───────────────────────────────────

from langchain_core.messages import BaseMessage
from typing import List

# Fonction d'approximation de tokens (4 chars ≈ 1 token)
# Utilisée car ChatOllama n'implémente pas get_num_tokens()
def approx_token_counter(messages: List[BaseMessage]) -> int:
    """Approximation : 4 caractères ≈ 1 token (règle empirique GPT)."""
    total = 0
    for msg in messages:
        # Compter le contenu textuel du message
        content = str(msg.content)
        total += len(content) // 4
    return total

# Trimmer par budget de tokens : max 200 tokens en contexte
token_trimmer = trim_messages(
    max_tokens=200,                      # budget max en tokens
    strategy="last",                     # garde les plus récents
    token_counter=approx_token_counter,  # notre fonction d'approximation
    include_system=True,
    allow_partial=False,
    start_on="human",
)

store_token = {}

def get_history_token(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store_token:
        store_token[session_id] = InMemoryChatMessageHistory()
    return store_token[session_id]

prompt_token = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant support. Budget contexte : 200 tokens max."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain_token = (
    RunnablePassthrough.assign(
        history=lambda x: token_trimmer.invoke(x["history"])
    )
    | prompt_token
    | llm
    | StrOutputParser()
)

chain_with_tokens = RunnableWithMessageHistory(
    chain_token,
    get_history_token,
    input_messages_key="input",
    history_messages_key="history",
)

# ─── Test avec inspection du budget utilisé ──────────────────────────
config_t = {"configurable": {"session_id": "token_001"}}

# Seed l'historique avec plusieurs échanges
seed_messages = [
    "Mon serveur web renvoie une erreur 502 Bad Gateway.",
    "J'ai vérifié les logs Nginx, je vois des timeouts upstream.",
    "Le service backend tourne bien, j'ai vérifié avec systemctl status.",
    "La base de données PostgreSQL est accessible et répond en moins de 10ms.",
]

print("=== Budget de tokens (max ~200 tokens en contexte) ===\n")
for q in seed_messages:
    r = chain_with_tokens.invoke({"input": q}, config=config_t)
    
    # Calculer le budget utilisé APRÈS trim
    hist = get_history_token("token_001")
    trimmed = token_trimmer.invoke(hist.messages)
    budget_used = approx_token_counter(trimmed)
    
    print(f"[H] {q[:60]}...")
    print(f"[S] {r[:100]}...")
    print(f"      Budget contexte : ~{budget_used} tokens ({len(trimmed)} messages conservés)\n")


---
## 🧩 Concept 4 — Résumé Automatique (Compression LCEL)

### 📖 Théorie

Au lieu de tronquer l'historique, on peut le **résumer** : condenser les N derniers messages en un paragraphe de synthèse, puis passer ce résumé comme contexte.

```
[M1, M2, M3, M4] → [Résumé LLM] → Prompt avec résumé → LLM → Réponse
```

C'est plus coûteux (un appel LLM supplémentaire) mais conserve l'**essence** de la conversation, pas juste la fin.

> **💡 Trainer's Note** : En production, on ne résume pas à chaque tour — seulement quand `len(history) > seuil`. On implémente ici une version simplifiée avec résumé systématique pour la clarté pédagogique.


In [8]:
# ─── Concept 4 : Résumé Automatique ─────────────────────────────────

from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate

# Prompt de compression/résumé
SUMMARY_PROMPT = PromptTemplate.from_template(
    """Résume cette conversation de support technique en 2-3 phrases.
Inclus : le problème signalé, les étapes déjà tentées, et l'état actuel.
Sois factuel et concis.

CONVERSATION:
{history_text}

RÉSUMÉ:"""
)

def summarize_history(messages: list) -> str:
    """Résume une liste de messages en texte compact."""
    if not messages:
        return ""
    
    # Formater les messages en texte lisible
    history_text = "\n".join([
        f"Utilisateur: {m.content}" if isinstance(m, HumanMessage) else f"Assistant: {m.content}"
        for m in messages
    ])
    
    # Appel LLM de compression (chaîne LCEL pure)
    summary_chain = SUMMARY_PROMPT | llm | StrOutputParser()
    return summary_chain.invoke({"history_text": history_text})

# Store pour ce concept
store_summary = {}

def get_history_summary(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store_summary:
        store_summary[session_id] = InMemoryChatMessageHistory()
    return store_summary[session_id]

# Prompt principal avec le résumé comme contexte
prompt_summary = ChatPromptTemplate.from_messages([
    ("system", 
     "Tu es un assistant support technique.\n"
     "CONTEXTE DE LA CONVERSATION (résumé): {summary}\n"
     "Utilise ce contexte pour répondre."),
    MessagesPlaceholder(variable_name="history"),  # dernier échange seulement
    ("human", "{input}"),
])

# Chaîne avec compression : résume l'historique ancien, garde le dernier échange
def build_chain_with_summary(session_id: str):
    """Construit dynamiquement la chaîne avec résumé de l'historique."""
    hist = get_history_summary(session_id)
    msgs = hist.messages
    
    if len(msgs) <= 2:
        # Pas assez d'historique → résumé vide, on garde tout
        summary = "Début de conversation."
        recent = msgs
    else:
        # Résumer tout sauf les 2 derniers messages
        to_summarize = msgs[:-2]
        recent = msgs[-2:]  # garder seulement le dernier échange
        summary = summarize_history(to_summarize)
    
    return summary, recent

# ─── Test ─────────────────────────────────────────────────────────────
session_id = "summary_001"
config_s = {"configurable": {"session_id": session_id}}

print("=== Résumé Automatique de l'historique ===\n")

# Alimenter l'historique manuellement pour la démo
hist = get_history_summary(session_id)
hist.add_user_message("Mon laptop ne démarre plus après une mise à jour Windows.")
hist.add_ai_message("Essayez de démarrer en mode sans échec (F8 au boot).")
hist.add_user_message("Le mode sans échec fonctionne, l'écran normal reste noir.")
hist.add_ai_message("Cela suggère un problème de pilote graphique. Désinstallez le dernier driver GPU.")
hist.add_user_message("J'ai désinstallé le driver, redémarré, mais l'écran est toujours noir.")

# Générer le résumé
summary, recent = build_chain_with_summary(session_id)

print(" Résumé de l'historique ancien :")
print(f"   {summary}\n")
print(f" Messages récents conservés : {len(recent)} message(s)\n")

# Poser une nouvelle question avec le résumé comme contexte
question = "Quelle est la prochaine étape diagnostique selon toi ?"
print(f"[H] {question}")

# Chaîne directe (sans RunnableWithMessageHistory pour cet exemple didactique)
chain_direct = prompt_summary | llm | StrOutputParser()
response = chain_direct.invoke({
    "summary": summary,
    "history": recent,
    "input": question,
})
print(f"[S] {response}")


=== Résumé Automatique de l'historique ===

📝 Résumé de l'historique ancien :
   L'utilisateur signale que son laptop ne démarre plus suite à une mise à jour Windows. Après avoir tenté de démarrer en mode sans échec, l'écran principal du laptop reste noir, mais le mode sans échec fonctionne. L'utilisateur est donc bloqué avec un écran noir après la mise à jour.

📌 Messages récents conservés : 2 message(s)

👤 Quelle est la prochaine étape diagnostique selon toi ?
🤖 Très bien. Si la désinstallation du pilote graphique n’a pas résolu le problème, il y a plusieurs pistes à explorer. 

Voici les prochaines étapes que nous pouvons essayer :

1. **Vérifiez les connexions :** Assurez-vous que tous les câbles (alimentation, écran) sont correctement connectés à votre laptop et au moniteur. Testez avec un autre câble si possible.

2. **Testez avec un autre écran (si possible) :** Si vous avez accès à un autre moniteur, connectez-le à votre laptop pour voir si l'image s'affiche. Cela permettra de 

---
## 🧩 Concept 5 — Mémoire Sémantique (ChromaDB Retriever dans LCEL)

### 📖 Théorie

La mémoire sémantique ne stocke pas l'historique chronologique mais des **faits** indexés par similarité vectorielle. On retrouve les souvenirs pertinents par **recherche sémantique**, pas par ordre temporel.

```
[Fait 1: bug imprimante] ──► Chroma ──► Query "problème réseau" ──► [Fait 3: réseau]
[Fait 2: CPU overheating]             (embedding similarity)
[Fait 3: réseau coupé]
```

C'est utile pour un **assistant qui se souvient de tous les tickets passés** d'un utilisateur, même vieux.

> **💡 Trainer's Note** : `VectorStoreRetrieverMemory` est remplacé par un retriever Chroma standard dans la chaîne LCEL. La logique "chercher avant de répondre" est explicitée, pas cachée dans une classe.


In [ ]:
# ─── Concept 5 : Mémoire Sémantique avec ChromaDB ───────────────────

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough

# ─── 1. Créer la base de mémoire vectorielle ─────────────────────────
# Simule l'historique des tickets précédents de cet utilisateur
past_tickets = [
    Document(
        page_content="Ticket #001: Imprimante HP LaserJet ne répond plus. Résolu en redémarrant le spouleur.",
        metadata={"ticket_id": "001", "status": "resolved", "date": "2024-01-15"}
    ),
    Document(
        page_content="Ticket #002: Surchauffe CPU Intel i7, températures à 95°C. Résolu en nettoyant les ventilateurs.",
        metadata={"ticket_id": "002", "status": "resolved", "date": "2024-02-10"}
    ),
    Document(
        page_content="Ticket #003: Connexion WiFi instable, déconnexions fréquentes. Résolu en mettant à jour le driver réseau.",
        metadata={"ticket_id": "003", "status": "resolved", "date": "2024-03-05"}
    ),
    Document(
        page_content="Ticket #004: Erreur 0x80070002 lors de Windows Update. Résolu avec l'outil DISM et SFC.",
        metadata={"ticket_id": "004", "status": "resolved", "date": "2024-03-20"}
    ),
    Document(
        page_content="Ticket #005: Écran bleu BSOD avec code IRQL_NOT_LESS_OR_EQUAL. Résolu en désinstallant le driver réseau tiers.",
        metadata={"ticket_id": "005", "status": "resolved", "date": "2024-04-01"}
    ),
]

# Indexer les tickets dans ChromaDB (in-memory pour la démo)
memory_db = Chroma.from_documents(
    documents=past_tickets,
    embedding=embeddings,
    collection_name="support_memory",
)

# ─── 2. Retriever : cherche les 2 tickets les plus similaires ────────
memory_retriever = memory_db.as_retriever(search_kwargs={"k": 2})

# ─── 3. Chaîne LCEL avec mémoire sémantique ──────────────────────────
def format_memory(docs) -> str:
    """Formate les documents récupérés en contexte lisible."""
    if not docs:
        return "Aucun ticket similaire trouvé."
    return "\n".join([
        f"- {d.page_content} (Ticket {d.metadata['ticket_id']})"
        for d in docs
    ])

prompt_semantic = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es un assistant support technique avec accès aux tickets précédents de l'utilisateur.\n\n"
     "TICKETS SIMILAIRES DANS L'HISTORIQUE:\n{memory_context}\n\n"
     "Utilise cet historique pour personnaliser ta réponse."),
    ("human", "{input}"),
])

# La chaîne récupère d'abord la mémoire sémantique, puis répond
chain_semantic = (
    RunnablePassthrough.assign(
        # Recherche sémantique sur l'input utilisateur
        memory_context=lambda x: format_memory(
            memory_retriever.invoke(x["input"])
        )
    )
    | prompt_semantic
    | llm
    | StrOutputParser()
)

# ─── Test ─────────────────────────────────────────────────────────────
test_queries = [
    "Mon PC devient brûlant et se met en veille tout seul.",
    "J'ai des coupures réseau répétées depuis hier.",
]

print("=== Mémoire Sémantique (tickets passés) ===\n")
for query in test_queries:
    # Voir quels tickets ont été récupérés
    retrieved = memory_retriever.invoke(query)
    print(f"[H] Requête : {query}")
    print(f"🔍 Tickets récupérés : {[d.metadata['ticket_id'] for d in retrieved]}")
    
    response = chain_semantic.invoke({"input": query})
    print(f"[S] Réponse : {response[:200]}...\n")


=== Mémoire Sémantique (tickets passés) ===

👤 Requête : Mon PC devient brûlant et se met en veille tout seul.
🔍 Tickets récupérés : ['003', '002']
🤖 Réponse : Bonjour ! Je comprends que votre PC devient très chaud et se met en veille de manière inattendue. Cela peut être très frustrant, et nous allons essayer de trouver une solution. 

En fonction de votre ...

👤 Requête : J'ai des coupures réseau répétées depuis hier.
🔍 Tickets récupérés : ['001', '002']


---
## 🧩 Concept 6 — État Structuré avec LangGraph (`StateGraph` + `TypedDict`)

### 📖 Théorie

LangGraph introduit une mémoire **structurée** : au lieu d'une simple liste de messages, l'état est un **dictionnaire typé** qu'on enrichit à chaque étape.

```
State = {
    messages: [M1, M2, M3],         ← historique standard
    entities: {"utilisateur": "Mike", "OS": "Windows 11"},  ← entités extraites
    current_issue: "Imprimante KO"   ← contexte métier
}
```

C'est la réponse moderne à `ConversationEntityMemory` — mais plus puissant car **entièrement personnalisable**.

> **💡 Trainer's Note** : LangGraph change de paradigme : on programme un **graphe d'états**, pas une chaîne. Chaque nœud lit et modifie l'état. C'est plus verbeux mais infiniment plus flexible.


In [ ]:
# ─── Concept 6 : État Structuré LangGraph ────────────────────────────

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
import json

# ─── 1. Définir l'état structuré ─────────────────────────────────────
class SupportState(TypedDict):
    messages: Annotated[list, add_messages]    # historique (append automatique)
    entities: dict                              # entités extraites
    issue_type: str                             # catégorie du problème

# ─── 2. Nœud d'extraction d'entités ──────────────────────────────────
def extract_entities_node(state: SupportState) -> dict:
    """Extrait les entités clés du dernier message utilisateur."""
    if not state["messages"]:
        return {}
    
    last_msg = state["messages"][-1].content
    
    # Prompt d'extraction d'entités (léger)
    extraction_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Extrais les entités techniques du message suivant au format JSON.\n"
         "Entités possibles: appareil, OS, marque, erreur_code, symptome.\n"
         "Réponds UNIQUEMENT avec du JSON valide, sans markdown.\n"
         "Exemple: {{\"appareil\": \"imprimante\", \"marque\": \"HP\"}}"),
        ("human", "{text}"),
    ])
    
    try:
        extraction_chain = extraction_prompt | llm | StrOutputParser()
        raw = extraction_chain.invoke({"text": last_msg})
        # Nettoyer et parser le JSON
        raw_clean = raw.strip().strip("```json").strip("```").strip()
        new_entities = json.loads(raw_clean)
        # Fusionner avec les entités existantes (on accumule)
        merged = {**state.get("entities", {}), **new_entities}
        return {"entities": merged}
    except (json.JSONDecodeError, Exception):
        return {}  # Si extraction échoue, on continue sans erreur

# ─── 3. Nœud LLM principal ───────────────────────────────────────────
def chat_node(state: SupportState) -> dict:
    """Génère la réponse en utilisant l'historique ET les entités extraites."""
    entities_str = json.dumps(state.get("entities", {}), ensure_ascii=False)
    
    prompt_structured = ChatPromptTemplate.from_messages([
        ("system",
         "Tu es un assistant support technique.\n"
         f"ENTITÉS CONNUES: {entities_str}\n"
         "Utilise ces informations pour personnaliser ta réponse."),
        MessagesPlaceholder(variable_name="messages"),
    ])
    
    chain = prompt_structured | llm
    response = chain.invoke({"messages": state["messages"]})
    return {"messages": [response]}  # add_messages s'occupe de l'append

# ─── 4. Construire le graphe ──────────────────────────────────────────
graph_builder = StateGraph(SupportState)

# Ajouter les nœuds
graph_builder.add_node("extract_entities", extract_entities_node)
graph_builder.add_node("chat", chat_node)

# Définir les transitions : START → extract → chat → END
graph_builder.add_edge(START, "extract_entities")
graph_builder.add_edge("extract_entities", "chat")
graph_builder.add_edge("chat", END)

# Compiler le graphe
support_graph = graph_builder.compile()

# ─── 5. Test ──────────────────────────────────────────────────────────
print("=== LangGraph : État Structuré avec Entités ===\n")

# État initial
initial_state = SupportState(
    messages=[HumanMessage(content="Mon PC HP EliteBook sous Windows 11 affiche l'erreur 0x80004005.")],
    entities={},
    issue_type=""
)

# Exécuter le graphe
result = support_graph.invoke(initial_state)

print(f"🔍 Entités extraites : {result['entities']}")
print(f"💬 Réponse : {result['messages'][-1].content[:200]}...\n")

# Tour 2 : nouveau message, les entités s'accumulent
state_2 = SupportState(
    messages=result["messages"] + [HumanMessage(content="J'ai redémarré mais l'erreur revient à chaque démarrage.")],
    entities=result["entities"],
    issue_type=""
)

result_2 = support_graph.invoke(state_2)
print(f"🔍 Entités cumulées : {result_2['entities']}")
print(f"💬 Réponse tour 2 : {result_2['messages'][-1].content[:200]}...")


---
## 🧩 Concept 7 — Graphe de Connaissances avec LangGraph

### 📖 Théorie

Un **graphe de connaissances** va plus loin que les entités : il stocke des **relations** entre faits. Par exemple : `"utilisateur" --possède--> "HP EliteBook"` et `"HP EliteBook" --tourne-sous--> "Windows 11"`.

En pratique dans notre stack locale, on implémente une version simplifiée : un dictionnaire de faits structurés qui s'enrichit à chaque échange.

> **💡 Trainer's Note** : Un vrai graphe de connaissances (triplets sujet/prédicat/objet) nécessite un LLM puissant pour l'extraction fiable. Avec `gemma3:4b` local, on reste sur un dictionnaire de faits — pragmatisme > hype.


In [ ]:
# ─── Concept 7 : Graphe de Connaissances (Facts Graph) ──────────────

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# ─── État avec graphe de faits ────────────────────────────────────────
class KGState(TypedDict):
    messages: Annotated[list, add_messages]
    facts: dict   # {sujet: {prédicat: objet}} — graphe simplifié
    summary: str  # résumé courant de la situation

# ─── Nœud d'extraction de faits ──────────────────────────────────────
def extract_facts_node(state: KGState) -> dict:
    """Extrait des faits structurés du dernier message."""
    if not state["messages"]:
        return {}
    
    last_msg = state["messages"][-1].content
    
    fact_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Extrais des faits techniques du message sous forme de triplets.\n"
         "Format JSON: {{\"sujet\": {{\"propriete\": \"valeur\"}}}}\n"
         "Exemple: {{\"machine\": {{\"modele\": \"HP EliteBook\", \"OS\": \"Windows 11\"}}}}\n"
         "UNIQUEMENT du JSON valide, sans markdown."),
        ("human", "{text}"),
    ])
    
    try:
        chain = fact_prompt | llm | StrOutputParser()
        raw = chain.invoke({"text": last_msg})
        raw_clean = raw.strip().strip("```json").strip("```").strip()
        new_facts = json.loads(raw_clean)
        
        # Fusionner en profondeur (deep merge)
        existing = state.get("facts", {})
        for subj, props in new_facts.items():
            if subj not in existing:
                existing[subj] = {}
            existing[subj].update(props)
        
        return {"facts": existing}
    except Exception:
        return {}

# ─── Nœud de génération de réponse ───────────────────────────────────
def respond_with_kg_node(state: KGState) -> dict:
    """Répond en utilisant le graphe de connaissances comme contexte."""
    # Formater le graphe de faits en texte
    kg_text = ""
    for subj, props in state.get("facts", {}).items():
        kg_text += f"  [{subj}]\n"
        for k, v in props.items():
            kg_text += f"    └─ {k}: {v}\n"
    
    if not kg_text:
        kg_text = "Aucun fait connu encore."
    
    prompt_kg = ChatPromptTemplate.from_messages([
        ("system",
         "Tu es un assistant support.\n\n"
         "GRAPHE DE CONNAISSANCES:\n" + (kg_text or "Vide") + "\n\n"
         "Utilise ces faits pour une réponse précise et personnalisée."),
        MessagesPlaceholder(variable_name="messages"),
    ])
    
    chain = prompt_kg | llm
    response = chain.invoke({"messages": state["messages"]})
    return {"messages": [response]}

# ─── Construire et compiler le graphe ────────────────────────────────
kg_builder = StateGraph(KGState)
kg_builder.add_node("extract_facts", extract_facts_node)
kg_builder.add_node("respond", respond_with_kg_node)
kg_builder.add_edge(START, "extract_facts")
kg_builder.add_edge("extract_facts", "respond")
kg_builder.add_edge("respond", END)

kg_graph = kg_builder.compile()

# ─── Test ─────────────────────────────────────────────────────────────
print("=== Graphe de Connaissances LangGraph ===\n")

# Conversation en 3 tours
conversation = [
    "J'ai un Dell XPS 15 avec Ubuntu 22.04 et un SSD NVMe de 512GB.",
    "Depuis hier, le SSD est à 98% de capacité et le système ralentit.",
    "J'ai des logs Docker qui prennent beaucoup de place, environ 40GB.",
]

current_state = KGState(messages=[], facts={}, summary="")

for msg_text in conversation:
    current_state["messages"] = current_state.get("messages", []) + [HumanMessage(content=msg_text)]
    result = kg_graph.invoke(current_state)
    
    print(f"[H] {msg_text}")
    print(f"🔍 Faits connus : {json.dumps(result['facts'], ensure_ascii=False)}")
    print(f"[S] {result['messages'][-1].content[:150]}...\n")
    
    # Mettre à jour l'état pour le tour suivant
    current_state = result


---
## 🧩 Concept 8 — Multi-Sessions avec Routing par `session_id`

### 📖 Théorie

En production, plusieurs utilisateurs parlent en **parallèle** au même assistant. Chaque utilisateur a son propre historique, complètement isolé.

`RunnableWithMessageHistory` gère ça nativement : le `session_id` sert de clé dans un store partagé. C'est aussi simple que ça.

```python
# Utilisateur A
chain.invoke({"input": "..."}, config={"configurable": {"session_id": "user_A"}})

# Utilisateur B (historique séparé)
chain.invoke({"input": "..."}, config={"configurable": {"session_id": "user_B"}})
```

> **💡 Trainer's Note** : En remplaçant le dict Python par une vraie BDD (Redis, PostgreSQL), ce pattern passe directement en production. La seule chose qui change est la fonction `get_session_history`.


In [ ]:
# ─── Concept 8 : Multi-Sessions ─────────────────────────────────────

# Store partagé entre toutes les sessions
shared_store: dict[str, InMemoryChatMessageHistory] = {}

def get_shared_history(session_id: str) -> InMemoryChatMessageHistory:
    """Retourne l'historique isolé pour chaque session_id."""
    if session_id not in shared_store:
        shared_store[session_id] = InMemoryChatMessageHistory()
        print(f"   📂 Nouvelle session créée : {session_id}")
    return shared_store[session_id]

# Prompt et chaîne standards
prompt_multi = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant support. Session : {session_id}."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain_multi = (
    RunnablePassthrough.assign(session_id=lambda x: x.get("session_id", "unknown"))
    | prompt_multi
    | llm
    | StrOutputParser()
)

# Note : on passe session_id dans l'input ET dans config
chain_with_sessions = RunnableWithMessageHistory(
    # Chaîne sans le session_id passthrough (sinon conflit)
    prompt_multi | llm | StrOutputParser(),
    get_shared_history,
    input_messages_key="input",
    history_messages_key="history",
)

# ─── Simulation : 3 utilisateurs en parallèle ────────────────────────
sessions = {
    "alice_001": [
        "Mon écran clignote après la mise à jour du driver.",
        "La fréquence de rafraîchissement était à 144Hz, maintenant c'est 60Hz.",
    ],
    "bob_002": [
        "Je ne peux pas me connecter au VPN de l'entreprise.",
        "Le certificat SSL est expiré selon le message d'erreur.",
    ],
    "carol_003": [
        "Outlook plante au démarrage depuis ce matin.",
        "J'ai un message OST file corrupted.",
    ],
}

print("=== Multi-Sessions : 3 utilisateurs en parallèle ===\n")
for session_id, questions in sessions.items():
    config = {"configurable": {"session_id": session_id}}
    print(f"─── Session : {session_id} ───")
    for q in questions:
        r = chain_with_sessions.invoke({"input": q}, config=config)
        print(f"  [H] {q[:60]}")
        print(f"  [S] {r[:100]}...\n")

# ─── Inspection du store partagé ─────────────────────────────────────
print("=== Store Partagé : Isolation des Sessions ===")
for sid, hist in shared_store.items():
    print(f"  {sid}: {len(hist.messages)} messages")
    
print("\n✅ Chaque session est complètement isolée dans le store partagé.")


---
## 🧩 Concept 9 — Persistance Cross-Session avec LangGraph Checkpointer

### 📖 Théorie

`RunnableWithMessageHistory` stocke l'historique en RAM — tout est perdu si le processus redémarre. Les **Checkpointers LangGraph** sauvegardent l'**état complet** du graphe de façon persistante.

- **`InMemorySaver`** : persistance en RAM (tests/démo)
- **`SqliteSaver`** : persistance sur disque (prod légère) — dans `langgraph-checkpoint-sqlite`

```python
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

# thread_id = identifiant unique de la conversation
config = {"configurable": {"thread_id": "ticket_42"}}
graph.invoke({"messages": [...]}, config=config)
```

> **💡 Trainer's Note** : `thread_id` ≈ `session_id` mais en LangGraph. La différence clé : le checkpointer sauvegarde **tout l'état** (`messages` + entités + faits…), pas juste les messages.


In [ ]:
# ─── Concept 9 : Persistance avec InMemorySaver ──────────────────────

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

# ─── État simple ──────────────────────────────────────────────────────
class SimpleState(TypedDict):
    messages: Annotated[list, add_messages]

# ─── Nœud LLM ─────────────────────────────────────────────────────────
def chat_node_persistent(state: SimpleState) -> dict:
    """Nœud de chat simple."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Tu es un assistant support. Tu te souviens de toute la conversation."),
        MessagesPlaceholder(variable_name="messages"),
    ])
    chain = prompt | llm
    response = chain.invoke({"messages": state["messages"]})
    return {"messages": [response]}

# ─── Graphe avec Checkpointer ─────────────────────────────────────────
builder = StateGraph(SimpleState)
builder.add_node("chat", chat_node_persistent)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# InMemorySaver : persistance en mémoire (survit aux appels, pas au restart)
checkpointer = InMemorySaver()
persistent_graph = builder.compile(checkpointer=checkpointer)

# ─── Simulation : conversation en 2 sessions distinctes ───────────────
# Thread 1 : ticket de Alice
config_alice = {"configurable": {"thread_id": "ticket_alice_007"}}

print("=== Session 1 : Alice (ticket_alice_007) ===\n")
r1 = persistent_graph.invoke(
    {"messages": [HumanMessage(content="Mon clavier Bluetooth ne se connecte plus à mon MacBook Pro.")]},
    config=config_alice # type: ignore
)
print(f"[H] Alice : Mon clavier Bluetooth ne se connecte plus à mon MacBook Pro.")
print(f"[S] {r1['messages'][-1].content[:200]}...\n")

# Thread 2 : ticket de Bob (état indépendant)
config_bob = {"configurable": {"thread_id": "ticket_bob_008"}}
r_bob = persistent_graph.invoke(
    {"messages": [HumanMessage(content="Mes emails ne se synchronisent plus dans Outlook 365.")]},
    config=config_bob
)
print(f"=== Session 2 : Bob (ticket_bob_008) ===\n")
print(f"[H] Bob : Mes emails ne se synchronisent plus dans Outlook 365.")
print(f"[S] {r_bob['messages'][-1].content[:200]}...\n")

# ─── Reprendre le thread d'Alice (persistance !) ──────────────────────
print("=== Reprise de la session Alice (même thread_id) ===\n")
r2 = persistent_graph.invoke(
    {"messages": [HumanMessage(content="J'ai essayé de ré-associer le clavier, mais il ne s'affiche même pas dans la liste.")]},
    config=config_alice  # ← même thread_id : LangGraph recharge l'état
)
print(f"[H] Alice (suite) : J'ai essayé de ré-associer le clavier, mais il ne s'affiche même pas.")
print(f"[S] {r2['messages'][-1].content[:200]}...\n")

# ─── Inspection de l'état sauvegardé ─────────────────────────────────
state_alice = persistent_graph.get_state(config_alice)
print(f"   État sauvegardé pour Alice : {len(state_alice.values['messages'])} messages dans le checkpoint")
print(f"   État sauvegardé pour Bob   : {len(persistent_graph.get_state(config_bob).values['messages'])} messages")


---
## 🔬 Concept 10 (Avancé) — Le "Prompt-as-Memory" : Mécanique Interne

### 📖 Théorie

Comprendre **pourquoi** la mémoire fonctionne est essentiel. Un LLM est **stateless** par nature : chaque appel est indépendant. La "mémoire" n'existe que dans le prompt.

```
[System] + [H: tour1] + [AI: réponse1] + [H: tour2] + [AI: réponse2] + [H: question_actuelle]
                                                                                ↑
                                          Tout ça est reconstruit à CHAQUE APPEL
```

La fenêtre de contexte est physiquement limitée (8K, 16K, 128K tokens selon le modèle). Quand elle est pleine, les messages anciens sont **tronqués** ou **perdus**.

> **💡 Trainer's Note** : Visualiser le prompt complet est le meilleur outil de débogage mémoire. `chain.invoke()` + inspection du `PromptValue` permet de voir exactement ce que le LLM reçoit.


In [ ]:
# ─── Concept 10 : Prompt-as-Memory — Visualisation Interne ──────────

from langchain_core.messages import HumanMessage, AIMessage

# ─── Visualiser le prompt complet envoyé au LLM ───────────────────────
# On utilise .invoke() sur le prompt seul (sans le LLM) pour inspecter

# Simuler un historique existant
fake_history = [
    HumanMessage(content="Mon imprimante HP ne répond plus."),
    AIMessage(content="Vérifiez que le spouleur d'impression est actif dans les services Windows."),
    HumanMessage(content="Le spouleur tourne, mais l'imprimante affiche 'hors ligne'."),
    AIMessage(content="Essayez de supprimer et réinstaller l'imprimante dans les paramètres."),
]

# Le prompt qui sera envoyé
prompt_debug = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant support technique."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# ─── Inspecter le prompt AVANT envoi au LLM ──────────────────────────
prompt_value = prompt_debug.invoke({
    "history": fake_history,
    "input": "J'ai réinstallé, mais elle est toujours hors ligne.",
})

print("=== PROMPT COMPLET ENVOYÉ AU LLM ===")
print(f"Nombre de messages dans le prompt : {len(prompt_value.messages)}\n")
for i, msg in enumerate(prompt_value.messages):
    role = type(msg).__name__.replace("Message", "")
    content_preview = msg.content[:80] + ("..." if len(msg.content) > 80 else "")
    print(f"[{i}] {role:15s} | {content_preview}")

# ─── Calculer l'espace mémoire utilisé ───────────────────────────────
print("\n=== ANALYSE DU BUDGET CONTEXTE ===")
total_chars = sum(len(m.content) for m in prompt_value.messages)
approx_tokens = total_chars // 4

print(f"Caractères totaux      : {total_chars}")
print(f"Tokens approximatifs   : ~{approx_tokens}")
print(f"Budget gemma3:4b       : ~8,192 tokens")
print(f"Utilisation            : ~{(approx_tokens/8192)*100:.1f}%")
print(f"Messages restants      : ~{(8192 - approx_tokens) // 100} échanges supplémentaires possibles")

# ─── Démontrer le stateless : deux appels indépendants ────────────────
print("\n=== DÉMONSTRATION : LLM est STATELESS ===")

q = "Quel est le problème que j'ai mentionné ?"

# Appel 1 : SANS historique → le LLM ne sait rien
r_sans = (prompt_debug | llm | StrOutputParser()).invoke({
    "history": [],  # historique vide !
    "input": q,
})
print(f"Sans historique : {r_sans[:120]}...")

# Appel 2 : AVEC historique → le LLM "se souvient"
r_avec = (prompt_debug | llm | StrOutputParser()).invoke({
    "history": fake_history,
    "input": q,
})
print(f"Avec historique : {r_avec[:120]}...")
print("\n→ La 'mémoire' est 100% dans le prompt. Le LLM lui-même ne stocke rien.")


---
## 🔬 Concept 11 (Avancé) — Threads & Checkpointers LangGraph

### 📖 Théorie

Les **Checkpointers** LangGraph sont le système de persistance natif pour les graphes d'états.

| Checkpointer | Stockage | Usage |
|---|---|---|
| `InMemorySaver` | RAM | Tests, démo |
| `SqliteSaver` | Fichier `.db` | Prod légère, mono-serveur |
| `PostgresSaver` | PostgreSQL | Prod scalable multi-serveur |

Un **thread** = une conversation identifiée par `thread_id`. LangGraph sauvegarde automatiquement chaque étape d'exécution, permettant le **rejeu** et la **reprise** après crash.

> **💡 Trainer's Note** : `InMemorySaver` → `SqliteSaver` est **un changement de deux lignes**. C'est le principe de l'architecture hexagonale : on change l'adaptateur, pas le cœur.


In [ ]:
# ─── Concept 11 : Threads & Checkpointers ───────────────────────────

from langgraph.checkpoint.memory import InMemorySaver
# Pour SqliteSaver : pip install langgraph-checkpoint-sqlite
# from langgraph.checkpoint.sqlite import SqliteSaver

# ─── Même graphe que concept 9, avec InMemorySaver ───────────────────
# (on réutilise persistent_graph déjà compilé)

# ─── Fonctionnalité avancée : historique des checkpoints ─────────────
print("=== Historique des Checkpoints d'un Thread ===\n")

# Créer un thread avec plusieurs interactions
thread_config = {"configurable": {"thread_id": "checkpoint_demo_001"}}

messages_to_send = [
    "Mon disque dur externe n'est plus reconnu sous Windows.",
    "Il n'apparaît pas dans le gestionnaire de périphériques.",
    "J'entends un clic au démarrage du disque.",
]

for msg in messages_to_send:
    persistent_graph.invoke(
        {"messages": [HumanMessage(content=msg)]},
        config=thread_config
    )

# ─── Lister tous les checkpoints sauvegardés ──────────────────────────
print("Checkpoints sauvegardés pour ce thread :")
checkpoints = list(persistent_graph.get_state_history(thread_config))
print(f"Nombre de checkpoints : {len(checkpoints)}")
for i, cp in enumerate(checkpoints):
    nb_msgs = len(cp.values.get("messages", []))
    step = cp.metadata.get("step", i)
    print(f"  Checkpoint {i+1}: step={step}, messages={nb_msgs}")

# ─── Reprendre depuis un checkpoint précédent (time-travel) ───────────
print("\n=== Time-Travel : Reprendre depuis le Checkpoint 2 ===\n")
if len(checkpoints) >= 2:
    # Récupérer l'état au checkpoint 2 (second état sauvegardé)
    cp_to_restore = checkpoints[-2]  # avant-dernier = après 2ème message
    
    # Reconfigurer avec ce checkpoint spécifique
    restore_config = {
        "configurable": {
            "thread_id": "checkpoint_demo_001",
            "checkpoint_id": cp_to_restore.config["configurable"]["checkpoint_id"]
        }
    }
    
    # Voir l'état à ce moment précis
    state_at_cp = persistent_graph.get_state(restore_config)
    print(f"État restauré : {len(state_at_cp.values['messages'])} messages")
    for msg in state_at_cp.values["messages"]:
        role = "[H]" if isinstance(msg, HumanMessage) else "[S]"
        print(f"  {role} {msg.content[:60]}...")
    
    print("\n→ En production : permet de rejouer une conversation depuis n'importe quel état.")
else:
    print("(Pas assez de checkpoints pour la démo time-travel)")

# ─── Comparaison InMemorySaver vs SqliteSaver ─────────────────────────
print("\n=== Passage à SqliteSaver (production légère) ===")
print("""
# ─── AVANT (tests) ────────────────────────────────────────────────
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# ─── APRÈS (production disque) ────────────────────────────────────
from langgraph.checkpoint.sqlite import SqliteSaver
# SqliteSaver crée automatiquement le fichier DB
with SqliteSaver.from_conn_string("./support_conversations.db") as checkpointer:
    graph = builder.compile(checkpointer=checkpointer)
    # Tout le reste du code est IDENTIQUE

# → Changement : 2 lignes. Architecture : inchangée.
""")


---
## 🔬 Concept 12 (Avancé) — Streamlit + `st.session_state`

### 📖 Théorie

Streamlit recrée le script Python **à chaque interaction utilisateur**. Sans gestion explicite, l'historique serait perdu à chaque clic.

**Pattern double-store** :
1. `st.session_state["history"]` → liste Python native pour afficher les bulles de chat
2. `st.session_state["lc_history"]` → `InMemoryChatMessageHistory` pour le contexte LangChain

> **💡 Trainer's Note** : Ne jamais instancier le LLM ou la chaîne en dehors du `if ... not in st.session_state` guard. Sinon, un nouveau LLM est créé à chaque rerender — des dizaines d'objets inutiles en RAM.


In [ ]:
# ─── Concept 12 : Code de l'interface Streamlit ─────────────────────
# Ce code est à mettre dans un fichier app.py et lancé avec :
#   streamlit run app.py

STREAMLIT_APP_CODE = '''
import streamlit as st
from langchain_ollama import ChatOllama
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import trim_messages

# ─── Initialisation (une seule fois par session) ──────────────────────
if "lc_history" not in st.session_state:
    # Store LangChain pour le contexte du LLM
    st.session_state["lc_history"] = InMemoryChatMessageHistory()
    # Historique d'affichage (liste de dicts {role, content})
    st.session_state["display_history"] = []
    
    # LLM et chaîne (instanciés UNE SEULE FOIS)
    llm = ChatOllama(model="gemma3:4b")
    
    trimmer = trim_messages(
        max_messages=10,  # fenêtre de 5 échanges
        strategy="last",
        token_counter=len,
        include_system=True,
        start_on="human",
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Tu es un assistant support technique local. Sois concis."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ])
    
    from langchain_core.runnables import RunnablePassthrough
    
    chain = (
        RunnablePassthrough.assign(history=lambda x: trimmer.invoke(x["history"]))
        | prompt
        | llm
        | StrOutputParser()
    )
    
    # Session_id fixe (une seule session par onglet)
    SESSION_ID = "streamlit_user"
    
    st.session_state["chain"] = RunnableWithMessageHistory(
        chain,
        lambda _: st.session_state["lc_history"],
        input_messages_key="input",
        history_messages_key="history",
    )
    st.session_state["session_id"] = SESSION_ID

# ─── Interface Streamlit ──────────────────────────────────────────────
st.title("🛠️ Assistant Support Technique Local")
st.caption(f"Modèle : gemma3:4b · Stack 100% locale")

# Afficher l'historique de la conversation
for msg in st.session_state["display_history"]:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Zone de saisie
if user_input := st.chat_input("Décrivez votre problème technique..."):
    # Afficher le message utilisateur
    st.session_state["display_history"].append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)
    
    # Générer la réponse avec streaming
    with st.chat_message("assistant"):
        with st.spinner("Analyse en cours..."):
            config = {"configurable": {"session_id": st.session_state["session_id"]}}
            response = st.session_state["chain"].invoke(
                {"input": user_input},
                config=config
            )
        st.markdown(response)
    
    # Sauvegarder pour l'affichage
    st.session_state["display_history"].append({"role": "assistant", "content": response})

# Bouton de reset
if st.button("🗑️ Nouvelle conversation"):
    st.session_state.clear()
    st.rerun()
'''

print("=== Code Streamlit (app.py) ===\n")
print(STREAMLIT_APP_CODE)

# Sauvegarder dans un fichier prêt à l'emploi
import os
output_path = "streamlit_memory_app.py"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(STREAMLIT_APP_CODE.strip())

print(f"\n✅ Fichier créé : {output_path}")
print("   Lancer avec : streamlit run streamlit_memory_app.py")


---
## 🔬 Concept 13 (Avancé) — Limites Pratiques et Implications

### 📖 Théorie

Connaître les **limites** est aussi important que les patterns. Trois limites critiques :

1. **Context Window** : `gemma3:4b` ≈ 8K tokens. Au-delà, les messages anciens sont silencieusement ignorés.
2. **Hallucinations de mémoire** : Le LLM peut "inventer" des échanges précédents s'il est biaisé.
3. **Latence accumulation** : Plus l'historique est long, plus l'inférence est lente (plus de tokens à traiter).

> **💡 Trainer's Note** : La règle d'or : **mesurer avant d'optimiser**. Utiliser `time.time()` pour profiler l'impact de la taille du contexte sur la latence.


In [ ]:
# ─── Concept 13 : Mesure des Limites ────────────────────────────────

import time
from langchain_core.messages import HumanMessage, AIMessage

# ─── Mesure 1 : Latence vs Taille du contexte ────────────────────────
print("=== Mesure : Latence en fonction de la taille du contexte ===\n")

# Générer des historiques de taille croissante
def generate_fake_history(n_exchanges: int) -> list:
    """Crée n échanges fictifs pour simuler un long historique."""
    msgs = []
    for i in range(n_exchanges):
        msgs.append(HumanMessage(content=f"Question de support numéro {i+1} : problème réseau."))
        msgs.append(AIMessage(content=f"Réponse technique {i+1} : vérifiez la configuration DNS et les règles firewall."))
    return msgs

prompt_bench = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant support."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
chain_bench = prompt_bench | llm | StrOutputParser()

# Tester avec différentes tailles d'historique
sizes = [0, 2, 5]  # n échanges (0, 2, 5)

print(f"{'Échanges':>10} | {'Tokens (~)':>12} | {'Latence (s)':>12}")
print("-" * 42)

for n in sizes:
    history = generate_fake_history(n)
    total_tokens = sum(len(m.content) // 4 for m in history)
    
    start = time.time()
    try:
        result = chain_bench.invoke({
            "history": history,
            "input": "Résume le problème en une phrase."
        })
        latency = time.time() - start
        print(f"{n:>10} | {total_tokens:>12} | {latency:>12.2f}s")
    except Exception as e:
        print(f"{n:>10} | {total_tokens:>12} | Erreur: {str(e)[:30]}")

# ─── Mesure 2 : Taille max pratique pour gemma3:4b ───────────────────
print("\n=== Fenêtre de contexte : gemma3:4b ===")
print("""
Modèle          : gemma3:4b (via Ollama)
Context window  : ~8,192 tokens (variable selon config Ollama)
Budget pratique : 6,000 tokens (laisser 2K pour la réponse)
Échanges max    : ~30 échanges courts / ~10 échanges détaillés

Règle empirique : 1 échange support = ~100-300 tokens
→ Sans trimming : saturation en 20-60 échanges
→ Avec trim(max_messages=10) : stabilité indéfinie
""")

# ─── Mesure 3 : Détecter une hallucination de mémoire ────────────────
print("=== Test : Hallucination de Mémoire ===\n")

# Historique intentionnellement vide
chain_test = (ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant support. Réponds uniquement sur la base de la conversation."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
]) | llm | StrOutputParser())

r_hallucination = chain_test.invoke({
    "history": [],  # historique VIDE
    "input": "Rappelle-moi quel problème j'avais mentionné au début de notre conversation.",
})

print(f"Question : 'Rappelle-moi quel problème j'avais mentionné...'")
print(f"Historique : VIDE")
print(f"Réponse : {r_hallucination[:200]}")
print("\n→ Observez si le LLM invente un historique (hallucination) ou admet qu'il n'y en a pas.")


---
## 🔬 Concept 14 (Avancé) — Stratégies de Production

### 📖 Théorie

En production, on combine plusieurs stratégies selon les contraintes métier :

| Stratégie | Coût | Précision | Quand l'utiliser |
|---|---|---|---|
| Fenêtre glissante | Faible | Perd le passé | Conversations courtes |
| Résumé + fenêtre | Moyen | Bonne | Conversations longues |
| Vectorstore | Moyen | Sémantique | Recherche dans le passé |
| LangGraph checkpointer | Faible | Excellente | Multi-session, prod |

**Pattern de production recommandé** : Fenêtre glissante + résumé déclenché au-delà d'un seuil.

> **💡 Trainer's Note** : Ne jamais over-engineer. Commencez avec la fenêtre glissante `trim_messages(max_messages=10)`. Ajoutez le résumé uniquement si les utilisateurs se plaignent de perte de contexte.


In [ ]:
# ─── Concept 14 : Stratégie de Production Combinée ─────────────────

# Pattern : Fenêtre glissante + Résumé automatique au-delà du seuil
# Décision : résumer si l'historique dépasse SUMMARY_THRESHOLD messages

SUMMARY_THRESHOLD = 6   # déclencher le résumé au-delà de 6 messages
WINDOW_KEEP = 4         # conserver les 4 derniers messages après résumé

class ProductionMemoryManager:
    """
    Gestionnaire de mémoire de production.
    Stratégie hybride : fenêtre glissante + résumé déclenché par seuil.
    """
    
    def __init__(self, threshold: int = 6, window: int = 4):
        self.threshold = threshold
        self.window = window
        self.store: dict[str, InMemoryChatMessageHistory] = {}
        self.summaries: dict[str, str] = {}  # résumé par session
        
        # LLM de résumé (même modèle local)
        self.summary_chain = (
            ChatPromptTemplate.from_template(
                "Résume cette conversation de support en 3 phrases max. "
                "Focus : problème, étapes tentées, état actuel.\n\n{history}"
            )
            | llm
            | StrOutputParser()
        )
        
        # Chaîne principale
        self.chat_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Tu es un assistant support.\n"
             "{summary_context}"  # résumé injecté si disponible
            ),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ])
        self.chat_chain = self.chat_prompt | llm | StrOutputParser()
    
    def get_or_create_session(self, session_id: str) -> InMemoryChatMessageHistory:
        if session_id not in self.store:
            self.store[session_id] = InMemoryChatMessageHistory()
        return self.store[session_id]
    
    def maybe_summarize(self, session_id: str) -> None:
        """Résume et compresse l'historique si le seuil est dépassé."""
        hist = self.get_or_create_session(session_id)
        
        if len(hist.messages) > self.threshold:
            # Messages à résumer (tout sauf les WINDOW_KEEP derniers)
            to_summarize = hist.messages[:-self.window]
            recent = hist.messages[-self.window:]
            
            # Construire le texte à résumer
            history_text = "\n".join([
                f"{'User' if isinstance(m, HumanMessage) else 'AI'}: {m.content}"
                for m in to_summarize
            ])
            
            # Générer le résumé
            new_summary = self.summary_chain.invoke({"history": history_text})
            
            # Mettre à jour : garder seulement les messages récents + nouveau résumé
            self.summaries[session_id] = new_summary
            hist.clear()
            for msg in recent:
                hist.add_message(msg)
            
            print(f"   🗜️ Compression : {len(to_summarize)} → résumé + {len(recent)} récents")
    
    def chat(self, session_id: str, user_input: str) -> str:
        """Invoke principal avec gestion automatique de la mémoire."""
        # 1. Récupérer/créer la session
        hist = self.get_or_create_session(session_id)
        
        # 2. Ajouter le message utilisateur
        hist.add_user_message(user_input)
        
        # 3. Vérifier si on doit résumer
        self.maybe_summarize(session_id)
        
        # 4. Préparer le contexte de résumé
        summary = self.summaries.get(session_id, "")
        summary_context = f"RÉSUMÉ CONTEXTUEL:\n{summary}\n" if summary else ""
        
        # 5. Générer la réponse
        response = self.chat_chain.invoke({
            "summary_context": summary_context,
            "history": hist.messages,
            "input": user_input,
        })
        
        # 6. Sauvegarder la réponse
        hist.add_ai_message(response)
        
        return response

# ─── Test de la stratégie de production ──────────────────────────────
manager = ProductionMemoryManager(threshold=6, window=4)
session = "prod_test_001"

long_conversation = [
    "Mon serveur Ubuntu ne démarre plus après une mise à jour apt-get.",
    "J'obtiens l'erreur 'Kernel panic - not syncing' au boot.",
    "J'ai essayé de booter sur un kernel précédent dans GRUB, même erreur.",
    "Le système de fichiers root est en ext4 sur un RAID 1 logiciel.",
    "J'ai lancé fsck depuis un live USB, il a corrigé des erreurs.",
    "Après fsck, le boot aboutit maintenant mais PostgreSQL ne démarre pas.",
    "L'erreur PostgreSQL est : 'could not open file pg_wal: No such file or directory'.",
    "Il semble que le répertoire pg_wal soit vide.",
]

print("=== Stratégie de Production : Seuil + Résumé Automatique ===\n")
print(f"Seuil de compression : {manager.threshold} messages")
print(f"Fenêtre conservée    : {manager.window} messages\n")

for i, q in enumerate(long_conversation, 1):
    hist_before = len(manager.get_or_create_session(session).messages)
    r = manager.chat(session, q)
    hist_after = len(manager.get_or_create_session(session).messages)
    
    summary_active = "✅ résumé actif" if session in manager.summaries else ""
    print(f"[Tour {i:02d}] Hist: {hist_before}→{hist_after} msgs {summary_active}")
    print(f"  [H] {q[:70]}...")
    print(f"  [S] {r[:100]}...\n")

print("\n=== État final ===")
print(f"Messages en mémoire : {len(manager.get_or_create_session(session).messages)}")
print(f"Résumé actif        : {manager.summaries.get(session, 'Non')[:100]}...")


---
##    Récapitulatif : Table de Décision

| Besoin | Pattern recommandé | API |
|--------|-------------------|-----|
| Chatbot simple | Buffer complet | `InMemoryChatMessageHistory` + `RunnableWithMessageHistory` |
| Contrôle taille | Fenêtre glissante | `trim_messages(strategy="last", max_messages=k)` |
| Contrôle budget | Budget tokens | `trim_messages(max_tokens=N)` |
| Longues sessions | Résumé auto | LCEL custom + seuil |
| Recherche passé | Sémantique | ChromaDB retriever |
| Entités métier | État structuré | `LangGraph StateGraph + TypedDict` |
| Multi-utilisateurs | Sessions isolées | Store dict + `session_id` |
| Persistance crash | Checkpointing | `InMemorySaver` → `SqliteSaver` |
| Production | Hybride | Fenêtre + résumé au seuil |

### ✅ Règles d'Or (production)
1. **Commencer simple** — `trim_messages(max_messages=10)` suffit dans 80% des cas
2. **Mesurer avant d'optimiser** — profiler la latence vs taille contexte
3. **Isoler les sessions** — ne jamais partager un store sans `session_id`
4. **Persister avec LangGraph** — `InMemorySaver` → `SqliteSaver` = 2 lignes à changer
5. **Zéro classe dépréciée** — tout ce notebook fonctionne sans `langchain_classic`
